# All-variable control (`fixed_p = 0`)

The control: train on a constant C10 (one cyclic group of order 10, held fixed every run) with
`fixed_p = 0`, so *every* element-symbol is reshuffled each run. With no stable symbol→element
mapping there is nothing to memorize, so the model can only solve **symbolically** (read this
run's mapping from the in-context examples).

Expect it to learn the algorithm and generalize to the held-out fact with **no grokking**: no
memorization shortcut means no memorize-first / generalize-late gap. That is the point of the
control — it shows grokking is *absent* when the shortcut is removed. Grokking proper needs a
shortcut *and* something to generalize to, which only happens at **intermediate `fixed_p`** (the
follow-up sweep). Weight decay is left on so `fixed_p` is the only thing that differs.

The readout is `symbolic_reliance` = how much held-out accuracy collapses when we relabel the
**context** symbols but leave the query intact. For an all-variable model it should rise *with*
`acc_full` and stay high — no late jump.

In [ ]:
import os, subprocess
# Make sure cwd is the repo root. Jupyter opens this notebook with cwd=experiments/, so step
# up; on a fresh Colab, clone the repo first.
if os.path.basename(os.getcwd()) == 'experiments':
    os.chdir('..')
if not os.path.exists('experiments/train_fixed_p.py'):
    if not os.path.isdir('algebra-grok'):
        subprocess.run(['git', 'clone', '-q',
                        'https://github.com/tohuya6/algebra-grok.git'], check=True)
    os.chdir('algebra-grok')
import torch
print('cwd:', os.getcwd())
print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '(cpu)')

## Run the control

The full p=0 control — 200k steps on the constant C10, the exact command the sweep uses for its
p=0 point. This takes a while (hours on an A100 / H100); on the cluster you would usually launch
it in a terminal and come back for the plot:

```bash
bash experiments/train_c10.sh 0.0
```

The cell below runs exactly that. Skip it if you already ran it in a terminal — the plot reads
the saved `outputs/c10-p0.0/metrics.json` either way.

In [ ]:
!bash experiments/train_c10.sh 0.0

## Read the dynamics

`train_c10.sh` writes `outputs/c10-p0.0/metrics.json` every 1000 steps. On a log-x axis:

- **acc_full** — held-out accuracy.
- **symbolic_reliance** — the generalization signal. For the all-variable control it should track
  `acc_full` upward and stay high, with **no late jump** (no grokking gap). While `acc_full` is
  still ~0 early on, `symbolic_reliance` is undefined/noisy; that is expected.

In [ ]:
import json
import matplotlib.pyplot as plt

h = [r for r in json.load(open('outputs/c10-p0.0/metrics.json')) if r['step'] > 0]
step = [r['step'] for r in h]

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(step, [r['acc_full'] for r in h])
ax[1].plot(step, [r['symbolic_reliance'] for r in h])
loss = [(r['step'], r['train_loss']) for r in h if r['train_loss'] is not None]
ax[2].plot([s for s, _ in loss], [l for _, l in loss])

for a, title, ylabel in zip(ax,
        ['acc_full (held-out accuracy)', 'symbolic_reliance', 'train_loss'],
        ['acc_full', 'reliance', 'loss']):
    a.set(title=title, xlabel='step', ylabel=ylabel)
    a.set_xscale('log'); a.grid(True, alpha=0.3)
plt.suptitle('fixed_p = 0 (all-variable control)')
plt.tight_layout(); plt.show()

## Next

This control establishes the no-grok baseline. To look for grokking, sweep into intermediate
`fixed_p`, where a memorization shortcut and held-out generalization coexist:

```bash
for p in 0.0 0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9; do bash experiments/train_c10.sh $p; done
```

then compare the `symbolic_reliance` trajectories against this `fixed_p=0` control.
`verify_solution.ipynb` covers the static readout (validated on the released reference model).